<h1><b>Modul Pembelajaran Informed Search RKA 25</b></h1>

In [114]:
from data.utils import idastar_search, rbfs_search, sma_star_search
import inspect

In [115]:
def psource(*functions):
    "Print the source code for the given function(s)."
    code = '\n\n'.join(inspect.getsource(fn) for fn in functions)
    print(code)


<h2><b>1. GBFS (Greedy Best-First Search)</b></h2>

![](data\GBFS.png)


Penjelasan Greedy Best-First Search (GBFS):
- GBFS adalah algoritma pencarian yang memilih node berdasarkan nilai heuristic (h(n)) terkecil
- Heuristic merepresentasikan perkiraan jarak dari node ke goal
- GBFS bersifat greedy, yaitu selalu memilih langkah yang tampak paling dekat ke tujuan saat ini
- Tidak mempertimbangkan biaya jalur yang sudah ditempuh (g(n))
- BFS dan GBFS berasal dari ide yang berbeda: BFS = traversal berbasis level dan GBFS = search berbasis heuristic
- Dalam implementasinya, GBFS menggunakan priority queue untuk selalu mengambil node dengan nilai heuristik terkecil. Di Python, struktur ini biasanya diimplementasikan dengan modul heapq


<h3><b>Heuristik Function</b></h3>

Heuristic adalah fungsi yang digunakan untuk memperkirakan jarak dari suatu node ke tujuan (goal).
Nilainya tidak harus akurat, tetapi digunakan untuk membantu algoritma memilih arah pencarian.

Pada Greedy Best-First Search: f(n) = h(n)

Artinya, algoritma hanya memilih node yang paling dekat ke goal berdasarkan estimasi.

<b></b>**What's a Heapq?**</b>

`heapq` adalah modul bawaan Python yang menyediakan **struktur data heap** , yaitu **priority queue** berbasis **binary heap** . Heap adalah pohon biner lengkap yang memenuhi sifat **heap property** :

* **Min-Heap** (default di`heapq`): Nilai terkecil selalu berada di root, sehingga`heapq.heappop()` selalu mengembalikan elemen terkecil.
* **Max-Heap** dapat dibuat dengan menyimpan nilai negatif dari elemen.

**Operasi Utama pada `heapq`** :

* `heapq.heappush(heap, item)`: Menambahkan`item` ke dalam heap.
* `heapq.heappop(heap)`: Menghapus dan mengembalikan elemen**terkecil** dari heap.
* `heapq.heappushpop(heap, item)`: Kombinasi`heappush` lalu`heappop`.
* `heapq.heapify(iterable)`: Mengubah iterable (list) menjadi heap dalam O(n).


Heap sangat berguna dalam **graph traversal** dan **shortest path , karena:

1. **Efisiensi dalam memilih elemen dengan prioritas tertinggi (bobot terkecil)** .
2. **Optimasi waktu pencarian node berikutnya** :
   * Menggunakan heap, kita mendapatkan**O(log n)** untuk**pengambilan node** dengan bobot terkecil.
   * Tanpa heap, pencarian node dengan bobot terkecil membutuhkan**O(n)** .
3. **Memastikan pemrosesan node dalam urutan optimal** (berdasarkan bobot jalur).





**Mekanisme setiap langkah GBFS:**

* Masukkan node awal ke priority queue berdasarkan ( h(n) )
* Ambil node dengan ( h(n) ) terkecil
* Jika goal → selesai
* Jika bukan → ekspansi neighbor dan masukkan ke queue (jika belum dikunjungi)
* Ulangi hingga goal ditemukan atau queue kosong


Kelebihan GBFS:
* Cepat dalam praktik (heuristic bagus) = Langsung menuju arah goal tanpa eksplorasi luas
* Sering lebih efisien daripada BFS, karena tidak mengeksplor semua node.
* Cocok untuk masalah dengan heuristic yang kuat = (misalnya pathfinding berbasis jarak)

Kekurangan GBFS:
* Tidak optimal = Tidak menjamin jalur terpendek
* Tidak selalu complete, terutama jika tidak ada mekanisme visited atau pada graph tertentu.
* Sangat bergantung pada heuristic = Heuristic buruk → performa buruk
* Bisa looping tanpa visited set = Karena tidak mempertimbangkan histori
* Mengabaikan biaya yang sudah ditempuh (g(n)) = Bisa memilih jalur pendek secara “visual” tapi mahal secara nyata

![](data\studi_case_GBFS.png)


In [116]:
graph = {
    'A': ['B', 'C', 'D'],
    'B': ['A', 'E'],
    'C': ['A', 'F', 'E'],
    'D': ['A', 'F'],
    'E': ['B', 'C', 'H'],
    'F': ['C', 'D', 'G'],
    'H': ['E', 'G'],
    'G': ['F', 'H']
}

In [117]:
# heuristik
h = {
    'A': 40,
    'B': 32,
    'C': 25,
    'D': 35,
    'E': 19,
    'F': 17,
    'H': 10,
    'G': 0
}

In [118]:
from heapq import heappush, heappop

def gbfs(graph, start, goal, h):
    pq = []
    heappush(pq, (h[start], start, [start]))

    visited = set()

    while pq:
        h_val, node, path = heappop(pq)

        if node == goal:
            return path

        if node not in visited:
            visited.add(node)

            for neighbor in graph[node]:
                if neighbor not in visited:
                    heappush(pq, (h[neighbor], neighbor, path + [neighbor]))

    return None

In [119]:
print(gbfs(graph, 'A', 'G', h))

['A', 'C', 'F', 'G']


<b>PR GBFS Algorithm</b>

![](data\PR_GBFS.png)


XZ-01 adalah robot kurir otonom yang dikembangkan untuk mengantarkan paket di sebuah kota berbentuk grid 5×5, di mana setiap sel merepresentasikan kondisi lingkungan yang bisa dilalui atau terhalang oleh rintangan. Robot ini harus bergerak dari titik awal menuju tujuan dengan memilih jalur seefisien mungkin, namun karena keterbatasan komputasi, ia tidak mengevaluasi seluruh kemungkinan jalur, melainkan menggunakan strategi yang selalu memilih langkah yang terlihat paling dekat ke tujuan berdasarkan estimasi. XZ-01 hanya dapat bergerak ke atas, bawah, kiri, atau kanan, sehingga ia harus menentukan sendiri fungsi estimasi jarak yang sesuai dengan pola pergerakan tersebut

petunjuk: pertimbangkan selisih posisi secara horizontal dan vertikal antara titik saat ini dan tujuan.

In [120]:
maze = [
    [0, 0, 0, 0, 0],
    [1, 1, 0, 1, 0],
    [0, 0, 0, 1, 0],
    [0, 1, 1, 0, 0],
    [0, 0, 0, 0, 0]
]

start = (0, 0)
goal = (4, 4)

In [121]:
from heapq import heappush, heappop

def heuristic(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def gbfs_maze(maze, start, goal):
    rows, cols = len(maze), len(maze[0])

    pq = []
    heappush(pq, (heuristic(start, goal), start, [start]))

    visited = set()

    while pq:
        h_val, node, path = heappop(pq)

        if node == goal:
            return path

        if node not in visited:
            visited.add(node)

            x, y = node
            moves = [(0,1), (1,0), (0,-1), (-1,0)]

            for dx, dy in moves:
                nx, ny = x + dx, y + dy

                neighbor = (nx, ny)

                if 0 <= nx < rows and 0 <= ny < cols and maze[nx][ny] == 0:
                    if neighbor not in visited:
                        heappush(
                            pq,
                            (heuristic(neighbor, goal), neighbor, path + [neighbor])
                        )

    return None

In [122]:
print('Path GBFS Maze:', gbfs_maze(maze, start, goal))

Path GBFS Maze: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 4), (2, 4), (3, 4), (4, 4)]



<h2><b>2. A* (A Star)</b></h2>

![](data\A_star.gif)


Algoritma **A\*** adalah metode **pencarian jalur terpendek** yang lebih efisien dibandingkan **Uniform Cost Search (UCS)** karena memanfaatkan **heuristik** untuk mempercepat pencarian solusi. Algoritma ini sering digunakan dalam **pathfinding** pada game, robotika, dan navigasi GPS.

Algoritma **A\*** bekerja dengan menggunakan **priority queue (heap)** untuk memilih simpul yang memiliki **total estimasi biaya terkecil** menuju **goal**. Estimasi ini dihitung dengan rumus:

f(n) = g(n) + h(n)

di mana:
- g(n) adalah **jarak aktual** dari **start** ke simpul saat ini.
- h(n) adalah **heuristik**, yaitu estimasi jarak dari simpul saat ini ke **goal**.


**Mekanisme setiap langkah A*:**

* Inisialisasi **open list** (priority queue) dengan node awal, dan **closed list** kosong

* Hitung nilai awal: ( f(n) = g(n) + h(n) ), dengan ( g(n) = 0 ) untuk start node

* Pilih node dengan nilai ( f(n) ) terkecil dari open list (node terbaik sementara)

* Jika node tersebut adalah goal → selesai (rekonstruksi path)

* Jika bukan:

  * Pindahkan node ke closed list
  * Ekspansi semua tetangganya (neighbor)

* Untuk setiap neighbor:

  * Hitung ( g(n) ) baru (biaya dari start ke node tersebut)
  * Hitung ( f(n) = g(n) + h(n) )
  * Jika node belum ada di open/closed → masukkan ke open list
  * Jika sudah ada tapi ditemukan jalur lebih murah → update nilai ( g(n) ) dan parent

* Ulangi proses dengan memilih node ( f(n) ) terkecil berikutnya dari open list

* Jika open list kosong dan goal tidak ditemukan → tidak ada solusi

Kelebihan A*:

* Optimal dan Complete = Menjamin menemukan jalur dengan cost terendah (jika heuristik admissible)
* Efisien secara waktu = Pencarian lebih terarah dibanding Dijkstra karena bantuan heuristik
* Fleksibel = Dapat digunakan di berbagai kasus dengan desain heuristik yang sesuai

Kekurangan A*:

* Boros memori = Menyimpan banyak node dalam open dan closed list
* Bergantung pada heuristik = Performa sangat tergantung kualitas h(n)

![](data\A_star_implementasi.png)


Seorang kurir ingin mengirim paket dari Kota A ke Kota J.

Setiap jalan memiliki biaya (jarak aktual), dan setiap kota memiliki
nilai heuristic (perkiraan jarak ke tujuan).

Gunakan algoritma A* untuk menentukan jalur tercepat dari A ke J.

In [123]:
# Representasi graf (format tuple)
graph = {
    'A': [('B', 6), ('F', 3)],
    'B': [('A', 6), ('C', 3), ('D', 2)],
    'C': [('B', 3), ('D', 1), ('E', 5)],
    'D': [('B', 2), ('C', 1), ('E', 8)],
    'E': [('C', 5), ('D', 8), ('I', 5), ('J', 5)],
    'F': [('A', 3), ('G', 1), ('H', 7)],
    'G': [('F', 1), ('I', 3)],
    'H': [('F', 7), ('I', 2)],
    'I': [('G', 3), ('H', 2), ('E', 5), ('J', 3)],
    'J': [('E', 5), ('I', 3)]
}

# Heuristik
h = {
    'A': 10, 'B': 8, 'C': 5, 'D': 7, 'E': 3,
    'F': 6,  'G': 5, 'H': 3, 'I': 1, 'J': 0
}

In [124]:
import heapq

def a_star(adj, start, goal, h):
    heap = [(h[start], 0, start, [start])]  # (f, g, node, path)
    g_score = {start: 0}

    while heap:
        f, g, node, path = heapq.heappop(heap)

        if g > g_score.get(node, float('inf')):
            continue

        # print(f"Mengunjungi: {node} | g={g}, f={f}")

        if node == goal:
            return g, path

        for neighbor, weight in adj[node]:
            new_g = g + weight
            if new_g < g_score.get(neighbor, float('inf')):
                g_score[neighbor] = new_g
                heapq.heappush(heap, (new_g + h[neighbor], new_g, neighbor, path + [neighbor]))

    return None

In [125]:
cost, path = a_star(graph, 'A', 'J', h)
print(f"Path terpendek : {' -> '.join(path)}")
print(f"Total cost     : {cost}")

Path terpendek : A -> F -> G -> I -> J
Total cost     : 10



<h3><b>Heuristik Function</b></h3>

Heuristic function adalah sebuah fungsi yang digunakan dalam algoritma pencarian untuk memperkirakan biaya atau jarak dari suatu node ke node tujuan. Fungsi ini tidak selalu memberikan hasil yang 100% akurat, tetapi cukup baik untuk membantu algoritma dalam menemukan jalur yang lebih optimal dan efisien.

Dalam Artificial Intelligence (AI) dan Graf, fungsi heuristic biasanya digunakan dalam algoritma seperti A (A-Star Search)* untuk membantu memilih jalur terbaik berdasarkan estimasi biaya.

Contoh _Heuristic Function_ yang bisa kita gunakan adalah sebagai berikut

![](data\heuristik.jpg)


**a. Zero Function Heuristic** <br>
Heuristic ini menganggap semua jarak estimasi ke tujuan bernilai 0, sehingga algoritma akan berjalan seperti Dijkstra.

In [126]:
def heuristic_zero():
    return 0

**b. Heuristic Manhattan Distance** <br>
Menggunakan selisih absolut koordinat x dan y antara dua titik.

In [127]:
def heuristic_manhattan(positions:dict, node, goal):
    x1, y1 = positions[node]
    x2, y2 = positions[goal]
    return abs(x1 - x2) + abs(y1 - y2)

**c. Heuristic Euclidean Distance** <br>
Euclidean Distance menghitung jarak garis lurus antara dua titik.

In [128]:
import math
def heuristic_euclidean(positions:dict, node, goal):
    x1, y1 = positions[node]
    x2, y2 = positions[goal]
    return math.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2)

<b> PR A* Algorithm </b>

![](data\PR_A_star.png)


Seorang pemain sedang menyelesaikan puzzle geser (8-puzzle).

Ia ingin mengubah kondisi awal menjadi kondisi goal dengan jumlah langkah minimum.

Gunakan algoritma A* dengan heuristic Manhattan Distance
untuk menemukan jalur solusi.

In [129]:
start = ((1,2,3),
         (0,4,6),
         (7,5,8))

GOAL = ((1,2,3),
        (4,5,6),
        (7,8,0))

In [130]:
# use manhattan distance
def heuristic(state):
    # Hitung total jarak manhattan setiap tile ke posisi tujuannya di GOAL
    dist = 0
    for i in range(3):
        for j in range(3):
            val = state[i][j]
            if val != 0:
                # Find the position of 'val' in the GOAL state
                for goal_row in range(3):
                    for goal_col in range(3):
                        if GOAL[goal_row][goal_col] == val:
                            goal_x = goal_row
                            goal_y = goal_col
                            break
                    else:
                        continue
                    break
                dist += abs(i - goal_x) + abs(j - goal_y)

    return dist

In [131]:
def puzzle_get_neighbors(state):
    neighbors = []

    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                x, y = i, j
    moves = [(0,1), (1,0), (0,-1), (-1,0)]
    for dx, dy in moves:
        nx, ny = x + dx, y + dy
        if 0 <= nx < 3 and 0 <= ny < 3:
            new_state = [list(row) for row in state]
            new_state[x][y], new_state[nx][ny] = new_state[nx][ny], new_state[x][y]
            neighbors.append(tuple(tuple(row) for row in new_state))
    return neighbors

In [132]:
import heapq

def a_star(start):
    heap = [(heuristic(start), 0, start, [start])]
    g_score = {start: 0}

    while heap:
        f, g, state, path = heapq.heappop(heap)
        if state == GOAL:
            return g, path
        if g > g_score.get(state, float('inf')):
            continue
        for neighbor in puzzle_get_neighbors(state):
            new_g = g + 1
            if new_g < g_score.get(neighbor, float('inf')):
                g_score[neighbor] = new_g
                heapq.heappush(heap,(new_g + heuristic(neighbor), new_g, neighbor, path + [neighbor]))
    return None

In [133]:
cost_8puzzle, path_8puzzle = a_star(start)
print(f"Path terpendek (8-puzzle): {path_8puzzle}")
print(f"Total cost (8-puzzle): {cost_8puzzle}")

Path terpendek (8-puzzle): [((1, 2, 3), (0, 4, 6), (7, 5, 8)), ((1, 2, 3), (4, 0, 6), (7, 5, 8)), ((1, 2, 3), (4, 5, 6), (7, 0, 8)), ((1, 2, 3), (4, 5, 6), (7, 8, 0))]
Total cost (8-puzzle): 3


<h2><b>3. Iterative Deepening A* (IDA*)</b></h2>
Iterative Deepening A* (IDA*) menggabungkan efisiensi penelusuran ruang memori dari Depth-First Search dengan tingkat optimalitas algoritma A*. Algoritma ini berjalan dalam sebuah loop berulang di mana setiap iterasinya menelusuri lintasan graf layaknya DFS, tetapi ia menetapkan batas biaya (<i>cost threshold</i>) berdasarkan f(n) = g(n) + h(n).

Mekanisme setiap langkah IDA*:
- Set threshold awal = nilai $f(n)$ dari root node
- Jalankan DFS, potong (cutoff) jalur jika nilai $f(n) >$ threshold
- Jika goal ditemukan → selesai
- Jika tidak → set threshold baru = nilai $f(n)$ minimum yang melampaui threshold sebelumnya
- Ulangi proses dengan threshold baru

Kelebihan IDA*:
- Efisien memori = $O(bd)$, meniru memori linier DFS
- Complete dan Optimal = Pasti menemukan cost terendah (jika heuristik admissible)

Kekurangan IDA*:
- Re-computation = Node yang sama sering dikunjungi berkali-kali pada setiap iterasi
- Overhead iterasi = Kurang efisien jika setiap node memiliki nilai heuristik yang unik


![](data\ida_star.gif)

Sumber GIF : https://algorithmsinsight.wordpress.com/wp-content/uploads/2016/03/ida-star.gif

<b>Implementasi IDA*</b>

Struktur Graf dengan heuristik:
- Graf memuat nilai pergerakan asli (g).
- Heuristik memuat batas bawah perkiraan sisa menuju <i>goal</i> (h).

In [134]:
psource(idastar_search)

def idastar_search(graph, heuristics, start, goal):
    threshold = heuristics[start]
    
    def search(node, g, threshold, path):
        f = g + heuristics[node]
        if f > threshold:
            return f, None
        if node == goal:
            return -1, path + [node]
            
        min_threshold = float('inf')
        for neighbor, cost in graph.get(node, []):
            if neighbor not in path:
                temp_curr, result_path = search(neighbor, g + cost, threshold, path + [node])
                if temp_curr == -1:
                    return -1, result_path
                if temp_curr < min_threshold:
                    min_threshold = temp_curr
        return min_threshold, None

    path = []
    while True:
        temp, result_path = search(start, 0, threshold, path)
        if temp == -1:
            return result_path, threshold
        if temp == float('inf'):
            return None, float('inf')
        threshold = temp



In [135]:
graph_ida = {
    'S': [('A', 2), ('B', 5)],
    'A': [('C', 2), ('D', 4)],
    'B': [('D', 1)],
    'C': [('G', 2)],
    'D': [('G', 3)],
    'G': []
}

heuristics_ida = {
    'S': 6, 'A': 4, 'B': 4, 'C': 2, 'D': 2, 'G': 0
}

print("IDA* Path:", idastar_search(graph_ida, heuristics_ida, 'S', 'G')[0])

IDA* Path: ['S', 'A', 'C', 'G']


<b>PR IDA* Algorithm</b>

Fino berada di koordinat (0, 0) dan harus menuju lokasi Naga di koordinat (4, 4) pada sebuah matriks grid 5x5. Grid ini memiliki rintangan bernilai 1 yang tidak dapat dilalui, sedangkan area bebas bernilai 0. Temukan jalur terpendek menggunakan algoritma IDA* dengan heuristik jarak Manhattan.

In [136]:
def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def get_neighbors(grid, r, c):
    neighbors = []
    for dr, dc in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < len(grid) and 0 <= nc < len(grid[0]) and grid[nr][nc] == 0:
            neighbors.append((nr, nc))
    return neighbors

def idastar_grid(grid, start, goal):
    def search(node, g, threshold, path_so_far):
        f = g + manhattan(node, goal)
        if f > threshold:
            return f, None
        if node == goal:
            return -1, path_so_far + [node]

        min_threshold = float('inf')
        for neighbor in get_neighbors(grid, node[0], node[1]):
            if neighbor not in path_so_far:
                temp, result_path = search(neighbor, g + 1, threshold, path_so_far + [node])
                if temp == -1:
                    return -1, result_path
                if temp < min_threshold:
                    min_threshold = temp
        return min_threshold, None

    threshold = manhattan(start, goal)
    while True:
        temp, result_path = search(start, 0, threshold, [])
        if temp == -1:
            return result_path
        if temp == float('inf'):
            return None
        threshold = temp

In [137]:
grid_1 = [
    [0, 0, 1, 0, 0],
    [0, 1, 0, 0, 1],
    [0, 0, 0, 1, 0],
    [1, 1, 0, 0, 0],
    [0, 0, 0, 1, 0]
]
path_ida = idastar_grid(grid_1, (0,0), (4,4))

In [138]:
print("IDA* Grid Path:", path_ida)

IDA* Grid Path: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (3, 2), (3, 3), (3, 4), (4, 4)]


<h2><b>4. Recursive Best-First Search (RBFS)</b></h2>
Recursive Best-first Search (RBFS) meniru perilaku standar pencarian Best-First Search namun menggunakan kapasitas memori linear yang menelusuri simpul paling menjanjikan secara rekursif, dilengkapi mekanisme backtracking untuk mengevaluasi jalur alternatif.

Mekanisme setiap langkah RBFS:
- Eksplorasi node mengikuti jalur dengan nilai $f(n)$ terkecil
- Simpan nilai $f(n)$ alternatif terbaik (node saudara) di setiap langkah
- Jika nilai $f(n)$ node saat ini melampaui nilai alternatif → lakukan backtrack
- Saat backtrack, perbarui nilai $f(n)$ node induk dengan nilai terbaik dari anak-anaknya
- Pindah eksplorasi ke jalur alternatif tersebut

Kelebihan RBFS:
- Efisien memori = Menggunakan ruang $O(bd)$
- Optimal = Selama heuristik yang digunakan admissible

Kekurangan RBFS:
- Thrashing (re-computation ekstrem) = Terlalu sering bolak-balik jalur pencarian
- Tidak memanfaatkan sisa memori = Tetap menggunakan sedikit memori meskipun tersedia besar

<b>Implementasi RBFS</b>

In [139]:
psource(rbfs_search)

def rbfs_search(graph, heuristics, start, goal):
    def rbfs(node, node_f, f_limit, path):
        if node == goal:
            return True, path + [node], node_f
            
        successors = graph.get(node, [])
        if not successors:
            return False, [], float('inf')
            
        succ_nodes = []
        for neighbor, cost in successors:
            total_g = len(path) * 1 
            child_f = max(total_g + cost + heuristics[neighbor], node_f)
            succ_nodes.append([child_f, neighbor, cost])
            
        while True:
            succ_nodes.sort(key=lambda x: x[0])
            best_f, best_node, best_cost = succ_nodes[0]
            
            if best_f > f_limit:
                return False, [], best_f
                
            alt_f = succ_nodes[1][0] if len(succ_nodes) > 1 else float('inf')
            
            result_bool, result_path, new_best_f = rbfs(
                best_node, best_f, min(f_limit, alt_f), path + [node]
      

In [140]:
graph_rbfs = {
    'N1': [('N2', 3), ('N3', 2)],
    'N2': [('N4', 4)],
    'N3': [('N4', 1), ('N5', 6)],
    'N4': [('N5', 2)],
    'N5': []
}

heuristics_rbfs = {
    'N1': 5, 'N2': 4, 'N3': 3, 'N4': 2, 'N5': 0
}

print("RBFS Path:", rbfs_search(graph_rbfs, heuristics_rbfs, 'N1', 'N5'))

RBFS Path: ['N1', 'N3', 'N4', 'N5']


**PR RBFS Algorithm**

Setelah bertemu di titik (4, 4), Naga dan Fino harus melanjutkan perjalanan menuju titik evakuasi di (0, 5) pada matriks perluasan 6x6. Area ini memiliki rintangan dan jarak antar simpul bernilai seragam. Gunakan algoritma RBFS untuk mengevaluasi batas nilai f(n) dan menemukan rute dengan biaya minimum secara rekursif.

In [141]:
def rbfs_grid(grid, start, goal):
    def rbfs(node, node_f, f_limit, path, g):
        if node == goal:
            return True, path + [node], node_f

        neighbors = get_neighbors(grid, node[0], node[1])
        if not neighbors:
            return False, [], float('inf')

        succ = []
        for nbr in neighbors:
            if nbr not in path:
                child_f = max(g + 1 + manhattan(nbr, goal), node_f)
                succ.append([child_f, nbr, 1])

        if not succ:
            return False, [], float('inf')

        while True:
            succ.sort(key=lambda x: x[0])
            best_f, best_node, best_cost = succ[0]

            if best_f > f_limit:
                return False, [], best_f

            alt_f = succ[1][0] if len(succ) > 1 else float('inf')
            res, res_path, best_f_new = rbfs(best_node, best_f, min(f_limit, alt_f), path + [node], g + best_cost)

            if res:
                return True, res_path, best_f
            succ[0][0] = best_f_new

    res, path, _ = rbfs(start, manhattan(start, goal), float('inf'), [], 0)
    return path if res else None

In [142]:
grid_2 = [
    [0, 0, 0, 1, 0, 0],
    [0, 1, 0, 0, 0, 0],
    [0, 1, 1, 1, 0, 1],
    [0, 0, 0, 1, 0, 0],
    [1, 1, 0, 0, 0, 0],
    [0, 0, 0, 1, 1, 0]
]
path_rbfs = rbfs_grid(grid_2, (4,4), (0,5))

In [143]:
print("RBFS Grid Path:", path_rbfs)

RBFS Grid Path: [(4, 4), (3, 4), (2, 4), (1, 4), (1, 5), (0, 5)]


<h2><b>5. Simplified Memory Bounded A* (SMA*)</b></h2>
Variasi algoritma A* yang memanfaatkan seluruh ruang memori tetap dengan menghapus node terburuk ketika kapasitas memori penuh.

Mekanisme setiap langkah SMA*:
- Jalankan A* biasa dengan mengekspansi node berbiaya $f(n)$ terkecil
- Jika batas memori tercapai → hapus (prune) leaf node dengan nilai $f(n)$ tertinggi
- Simpan (backup) nilai node yang dihapus ke node induk
- Jika diperlukan, node dapat diregenerasi kembali
- Ulangi proses

Kelebihan SMA*:
- Penggunaan memori optimal = Memanfaatkan memori maksimum yang tersedia
- Complete = Jika memori cukup untuk menyimpan jalur solusi
- Optimal = Jika memori cukup untuk menyimpan solusi terbaik

Kekurangan SMA*:
- Overhead waktu sangat tinggi = Banyak proses hapus dan regenerasi node
- Rentan thrashing = Jika batas memori terlalu kecil dibanding kompleksitas masalah

<b>Implementasi SMA*</b>

In [144]:
psource(sma_star_search)

def sma_star_search(graph, heuristics, start, goal, max_memory=5):
    open_list = [(heuristics[start], 0, [start])]
    
    while open_list:
        open_list.sort(key=lambda x: (x[0], -len(x[2])))
        current_f, current_g, path = open_list.pop(0)
        current_node = path[-1]
        
        if current_node == goal:
            return path, current_f
            
        successors = graph.get(current_node, [])
        for neighbor, cost in successors:
            if neighbor not in path:
                g_new = current_g + cost
                f_new = max(current_f, g_new + heuristics[neighbor])
                open_list.append((f_new, g_new, path + [neighbor]))
                
        if len(open_list) > max_memory:
            open_list.sort(key=lambda x: (x[0], -len(x[2])))
            open_list.pop() 
            
    return None, float('inf')



In [145]:
graph_sma = {
    'R1': [('R2', 1), ('R3', 2)],
    'R2': [('R4', 3)],
    'R3': [('R4', 1), ('R5', 2)],
    'R4': [('R6', 2)],
    'R5': [('R6', 3)],
    'R6': []
}

heuristics_sma = {
    'R1': 4, 'R2': 3, 'R3': 3, 'R4': 2, 'R5': 2, 'R6': 0
}

print("SMA* Path:", sma_star_search(graph_sma, heuristics_sma, 'R1', 'R6', max_memory=4)[0])

SMA* Path: ['R1', 'R3', 'R4', 'R6']


<b>PR SMA* Algorithm</b>

Setelah evakuasi, Naga dan Fino memasuki labirin terakhir dengan ukuran 5x5 dari titik (0, 0) menuju (4, 4). Sistem navigasi mereka mengalami keterbatasan sehingga memori yang tersedia hanya mampu menampung batas maksimal 6 simpul secara bersamaan. Lakukan pencarian jalur yang mematuhi batasan memori tersebut menggunakan algoritma SMA*.

In [146]:
def sma_star_grid(grid, start, goal, max_mem=6):
    open_list = [(manhattan(start, goal), 0, [start])]

    while open_list:
        open_list.sort(key=lambda x: (x[0], -len(x[2])))
        curr_f, curr_g, path = open_list.pop(0)
        curr_node = path[-1]

        if curr_node == goal:
            return path

        for nbr in get_neighbors(grid, curr_node[0], curr_node[1]):
            if nbr not in path:
                g_new = curr_g + 1
                f_new = max(curr_f, g_new + manhattan(nbr, goal))
                open_list.append((f_new, g_new, path + [nbr]))

        if len(open_list) > max_mem:
            open_list.sort(key=lambda x: (x[0], -len(x[2])))
            open_list.pop()

    return None

In [147]:
grid_3 = [
    [0, 0, 0, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 0, 1, 0],
    [1, 1, 0, 0, 0],
    [0, 0, 0, 1, 0]
]
path_sma = sma_star_grid(grid_3, (0,0), (4,4), max_mem=6)

In [148]:
print("SMA* Grid Path:", path_sma)

SMA* Grid Path: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 4), (2, 4), (3, 4), (4, 4)]
